[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/04_cell_specific_six_sweep_fitting.ipynb)

## Colab setup and Step 04 run controls

Run the first setup cell before any imports. In Google Colab it clones this repository, installs `requirements.txt` (including Optuna), changes into the repository root, and puts the repo on `sys.path` so `src.*` imports work.

### Reviewer-facing target scope

By default, Step 04 uses the full available ATF cell set and six current sweeps per cell:

- `ASTROMODEL_STEP04_SELECTED_FILE_IDS=all` -> `selected_file_ids=None`.
- `ASTROMODEL_STEP04_MAX_CELLS=all` -> no cell cap.
- `ASTROMODEL_STEP04_FORCE_RERUN=0` allows reuse of an already completed validated full run when present.
- `ASTROMODEL_STEP04_FORCE_RERUN=1` reruns the full optimizer path.

Small subsets are not a reviewer-facing default. Use them only for explicit method-development or generalization experiments, and label the output directory accordingly.

Step 04 fits the six current sweeps for each selected cell. There is no separate region selector in this notebook; to run a deliberate development subset, pass comma-separated file IDs through `ASTROMODEL_STEP04_SELECTED_FILE_IDS`. `ASTROMODEL_STEP04_N_FIT_POINTS` controls trace downsampling for speed/accuracy, not which current sweeps are included.

### Optimizer and loss controls

Use these environment variables before the Step 04 run cell:

- `ASTROMODEL_STEP04_OPTIMIZER_BACKEND`: `hybrid` (default reviewer-facing), `least_squares`, `optuna_scalar`, or `optuna_multi`.
- `ASTROMODEL_STEP04_OPTUNA_OBJECTIVE`: `acceptance_margin` (default reviewer-facing), `metric_scalar`, or trace-shape variants.
- `ASTROMODEL_STEP04_OPTUNA_N_TRIALS`: number of Optuna trials for Optuna/hybrid backends.
- `ASTROMODEL_STEP04_OPTUNA_SAMPLER`: `tpe`, `random`, or `nsga2`; multi-objective runs default to NSGA-II when left as `tpe`.
- `ASTROMODEL_STEP04_RUN_HOLDOUT`: `1`/`true`/`yes` to run leave-one-sweep-out validation.
- `ASTROMODEL_STEP04_TRACE_LOSS_TYPE`: `COMBINED` (historical/default), `L2`, `L1`, `HUBER`, or `LOG_COSH`.
- `ASTROMODEL_STEP04_FEATURE_SET`: `primary_no_redundant` by default, so redundant features from Step 02 do not dominate the loss.

### Canonical outputs

Canonical downstream files are written under `outputs/cell_fits/`, including full candidate history in `cell_fit_candidates.csv`, accepted ensembles in `accepted_cell_ensembles.csv`, held-out screens in `heldout_current_screen.csv`, and a SQLite audit database when the run path generated one.

In [ ]:
# Colab / local repository setup
from pathlib import Path
import os, shutil, subprocess, sys


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


def _run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


REPO_URL = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
REPO_BRANCH = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
PROJECT_DIRNAME = os.environ.get("ASTROMODEL_PROJECT_DIRNAME", "astromodel_proving")

# In Colab, set these before running the notebook if you want Drive persistence:
# os.environ["ASTROMODEL_STEP04_OUTPUT_DIR"] = "/content/drive/MyDrive/astromodel_outputs/step04_full"
# os.environ["ASTROMODEL_STEP04_BACKUP_DIR"] = "/content/drive/MyDrive/astromodel_outputs/step04_backups"
# os.environ["ASTROMODEL_STEP04_RUN_LABEL"] = "least_squares_full_2026_05"

if _running_in_colab():
    project_root = Path("/content") / PROJECT_DIRNAME
    if not project_root.exists():
        try:
            _run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(project_root)])
        except subprocess.CalledProcessError:
            _run(["git", "clone", "--depth", "1", REPO_URL, str(project_root)])
    os.chdir(project_root)
    requirements = project_root / "requirements.txt"
    if requirements.exists():
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    project_root = next((c for c in candidates if (c / "src").is_dir() and (c / "data").is_dir()), current)
    os.chdir(project_root)

os.environ["ASTROMODEL_PROJECT_ROOT"] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Optional Colab/Drive data hook. If the repository clone does not contain
# data/2_K+ Pumps Data, set ASTROMODEL_DATA_DIR to a mounted directory with
# that ATF data before running this cell.
expected_data_dir = project_root / "data" / "2_K+ Pumps Data"
external_data_dir = os.environ.get("ASTROMODEL_DATA_DIR")
if external_data_dir and not expected_data_dir.exists():
    src_data = Path(external_data_dir).expanduser().resolve()
    expected_data_dir.parent.mkdir(parents=True, exist_ok=True)
    try:
        expected_data_dir.symlink_to(src_data, target_is_directory=True)
    except OSError:
        shutil.copytree(src_data, expected_data_dir)
if not expected_data_dir.exists():
    raise FileNotFoundError(
        f"Missing Step 04 ATF data at {expected_data_dir}. In Colab, mount Drive "
        "or upload data, then set ASTROMODEL_DATA_DIR to the directory containing the ATF files."
    )

print(f"ASTROMODEL_PROJECT_ROOT={project_root}")
print(f"Working directory={Path.cwd()}")
print(f"data exists={(project_root / 'data').exists()}, src exists={(project_root / 'src').exists()}")

# Step 04 — Cell-specific six-sweep fitting and accepted ensemble construction

This notebook validates Step 04 using the **expected reviewer-facing astrocyte ODE model**.

Scope of this notebook:
- verify that the implemented `src.astro_model.model` matches the expected equations;
- build cell-specific six-sweep fits under that model;
- use Step 02 region-aware thresholds to define accepted ensembles;
- run held-out-sweep screening as part of the reviewer-facing contract.

Claim boundary:
- this notebook creates accepted cell-specific ensembles;
- it does **not** by itself claim biological degeneracy;
- mechanistic decomposition belongs to Step 05;
- predictive robustness beyond held-out sweeps belongs to Step 06.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
import numpy as np
import pandas as pd
from IPython.display import display

from src.astro_model import build_paramdict, model
from src.step04_cell_fits import acceptance_contract_table, load_step02_outputs_or_run

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
PROJECT_ROOT = Path(os.environ.get('ASTROMODEL_PROJECT_ROOT', Path.cwd())).resolve()

# Notebook-only audit/report outputs are separate from canonical Step 04 outputs.
NOTEBOOK_OUTPUT_DIR = Path(os.environ.get(
    'ASTROMODEL_STEP04_NOTEBOOK_OUTPUT_DIR',
    PROJECT_ROOT / 'outputs' / 'cell_fits_step04_model_aligned_demo',
)).resolve()
STEP04_OUTPUT_DIR = Path(os.environ.get(
    'ASTROMODEL_STEP04_OUTPUT_DIR',
    PROJECT_ROOT / 'outputs' / 'cell_fits',
)).resolve()
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEP04_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR
PROJECT_ROOT

## Model-alignment audit

In [ ]:
def reference_model(z, t, paramdict):
    Cm_a  = paramdict["Astrocyte"]["Cm_a"]
    g_kir = paramdict["Astrocyte"]["g_kir"]
    A = paramdict["Astrocyte"]["A"]
    g_k_a = paramdict["Astrocyte"]["g_k_a"]
    gl_a = paramdict["Astrocyte"]["gl_a"]
    w_a = paramdict["Astrocyte"]["w_a"]
    K_a0 = paramdict["Astrocyte"]["K_a0"]
    Sig_a = paramdict["Astrocyte"]["Sig_a"]
    gama_t = paramdict["Astrocyte"]["gama_t"]
    gama_s = paramdict["Astrocyte"]["gama_s"]
    Z_th = paramdict["Astrocyte"]["Z_th"]
    Z_s = paramdict["Astrocyte"]["Z_s"]
    Va_0 = paramdict["Astrocyte"]["Va_0"]
    Va_s = paramdict["Astrocyte"]["Va_s"]
    Va_l = paramdict["Astrocyte"]["Va_l"]
    P_k = paramdict["Astrocyte"]["P_k"]
    d_gap = paramdict["Astrocyte"]["d_gap"]
    F = paramdict["Astrocyte"]["F"]
    R = paramdict["Astrocyte"]["R"]
    T = paramdict["Astrocyte"]["T"]
    K_o0 =paramdict["external"]["K_o0"]
    w_o = paramdict["external"]["w_o"]
    epsilon = paramdict["external"]["epsilon"]
    idx = np.where(paramdict["external"]["K_bath"]["time"]<=t)[0][-1]
    K_bath = paramdict["external"]["K_bath"]["value"][idx]
    switching_function = paramdict["Astrocyte"].get("switching_function", "sigmoid")
    if "epsilon_middle" in paramdict["external"] and idx == 1:
      epsilon = epsilon*paramdict["external"]["epsilon_middle"]
    if "w_o_middle" in paramdict["external"] and idx == 1:
      w_o = w_o*paramdict["external"]["w_o_middle"]
    Va  = z[0]
    DK_a_t = z[1]
    K_a_s = z[2]
    Kg = z[3]
    DK_a = DK_a_t + K_a_s
    K_a  = K_a0 +DK_a
    DK_o_a = -(w_a/w_o)*DK_a_t
    K_o  = K_o0 + DK_o_a + Kg
    K_ratio = K_o / K_a
    if K_ratio <= 0: K_ratio = 1e-8
    E_k_a = 25.7 * np.log(K_ratio)
    I_k_a = g_k_a*(Va - E_k_a)
    I_Kir = g_kir * np.sqrt(np.abs(K_o))*(Va - E_k_a)*(1/(1+np.exp((Va - E_k_a)/19.2)))
    PH_a = 0.04*(Va - Va_s)
    P_kgap = d_gap*P_k
    exp_neg_PH_a = np.exp(-PH_a)
    denominator = -1 + np.exp(-PH_a)
    if denominator == 0: denominator = 1e-8
    I_kgap = P_kgap * F * PH_a * (1 / denominator) * ((K_a * exp_neg_PH_a) - K_a0)
    I_l_a  = gl_a*(Va - Va_l)
    if switching_function == "sigmoid":
        Th_s = DK_a / (1 + np.exp((Z_th - DK_a_t) * Z_s))
    elif switching_function == "tanh":
        Th_s = DK_a * (0.5 * (1 + np.tanh((DK_a_t - Z_th) * Z_s)))
    elif switching_function == "hill":
        n = paramdict["Astrocyte"].get("hill_coefficient", 2)
        K_d = paramdict["Astrocyte"].get("K_d", 1)
        Th_s = DK_a * ((DK_a_t ** n) / (K_d ** n + DK_a_t ** n))
    else:
        raise ValueError(f"Unknown switching function type: {switching_function}")
    dVa   = (-1.0/Cm_a)*(I_Kir + I_k_a +I_l_a +I_kgap)
    dDK_a_t = -(gama_t*Sig_a/(w_a*F))*(I_Kir + I_k_a)
    dK_a_s = -Th_s*(gama_s*Sig_a/(w_a*F))* I_kgap
    dKg   =  epsilon*(K_bath-K_o)
    return np.asarray([dVa,dDK_a_t,dK_a_s,dKg], dtype=float)

probe_cases = [
    ('CONTROL', 75, {'gki': 90.0, 'pk': 3e-4, 'd': 0.05, 'gt': 2.0, 'gs': 10.0, 'zth': 70.0, 'zs': 2.5, 'eps': 0.002, 'eps_middle': 1.0, 'wo': 1400.0, 'wo_middle': 1.0, 'ca': 500.0, 'gl_a': 5.0, 'Va_l': -70.0, 'Va_s': -92.0, 'switching_function': 'sigmoid', 'w_a': 2000.0}),
    ('MFA', 125, {'gki': 40.0, 'pk': 5e-5, 'd': 1.5, 'gt': 4.0, 'gs': 22.0, 'zth': 0.2, 'zs': 0.05, 'eps': 0.01, 'eps_middle': 0.8, 'wo': 2500.0, 'wo_middle': 1.0, 'ca': 400.0, 'gl_a': 0.01, 'Va_l': -70.0, 'Va_s': -90.0, 'switching_function': 'tanh', 'w_a': 2000.0}),
    ('MFA_BA', 100, {'gki': 25.0, 'pk': 2e-4, 'd': 1.5, 'gt': 7.0, 'gs': 14.0, 'zth': 0.2, 'zs': 0.05, 'eps': 0.01, 'eps_middle': 0.8, 'wo': 1700.0, 'wo_middle': 1.0, 'ca': 400.0, 'gl_a': 0.01, 'Va_l': -70.0, 'Va_s': -90.0, 'switching_function': 'hill', 'hill_coefficient': 3.0, 'K_d': 1.2, 'w_a': 2000.0}),
]
z = np.array([-80.0, 0.5, 0.2, 0.1], dtype=float)
probe_rows = []
for exp_type, current_na, flat in probe_cases:
    pdict = build_paramdict(exp_type, current_na, flat)
    deltas = []
    for t in [0.0, 11173.0, 12000.0, 21140.0, 22000.0]:
        got = model(z, t, pdict)
        ref = reference_model(z, t, pdict)
        deltas.append(float(np.max(np.abs(got - ref))))
    probe_rows.append({'condition': exp_type, 'current_na': current_na, 'max_abs_rhs_delta': max(deltas), 'status': 'exact_within_float_tolerance' if max(deltas) <= 1e-12 else 'mismatch'})
probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(OUTPUT_DIR / 'model_alignment_probe.csv', index=False)
display(probe_df)
assert (probe_df['status'] == 'exact_within_float_tolerance').all()

## Step 02 contract carried into Step 04

In [ ]:
step02_outputs = load_step02_outputs_or_run(PROJECT_ROOT, reuse_existing=True)
region_counts = step02_outputs['region_condition_cell_counts']
display(region_counts)
contract = acceptance_contract_table()
display(contract)

## Runtime-safe Step 04 fit run

By default this executed-review cell uses the full 37-cell target scope through `ASTROMODEL_STEP04_SELECTED_FILE_IDS=all`. A comma-separated `ASTROMODEL_STEP04_SELECTED_FILE_IDS` value is reserved for explicit development subsets, not reviewer-facing evidence.

In [ ]:
import time
import warnings
import shutil

from scipy.integrate import ODEintWarning
from src.atf_io import load_all_cells
from src.step04_cell_fits import run_step04_cell_specific_six_sweep_fitting
from src.step04_loss import Step04OptimizerConfig, Step04LossConfig, TraceLossConfig
from src.step04_outputs import save_step04_run_snapshot

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ModuleNotFoundError:
    optuna = None

warnings.filterwarnings("ignore", category=ODEintWarning)


def _env_int_or_none(name, default="all"):
    raw = os.environ.get(name, default).strip().lower()
    if raw in {"", "none", "all"}:
        return None
    return int(raw)


def _env_int(name, default):
    return int(os.environ.get(name, str(default)))


def _env_bool(name, default="0"):
    return os.environ.get(name, str(default)).strip().lower() in {"1", "true", "yes"}


def _env_worker_count(name, default):
    raw = os.environ.get(name, str(default)).strip().lower()
    return "auto" if raw in {"", "auto", "all"} else int(raw)


def _select_file_ids_per_group(project_root, per_group):
    if per_group is None:
        return None
    cells = load_all_cells(project_root / "data" / "2_K+ Pumps Data")
    groups = {}
    for cell in cells:
        groups.setdefault((cell.condition, cell.region), []).append(cell.file_id)
    selected = []
    for key in sorted(groups):
        selected.extend(sorted(groups[key])[:per_group])
    return selected


def _resolve_selected_file_ids():
    raw = os.environ.get("ASTROMODEL_STEP04_SELECTED_FILE_IDS", "all").strip()
    if raw.lower() in {"", "none", "all"}:
        return None
    if raw.lower() in {"group_balanced", "per_group", "two_per_group"}:
        return _select_file_ids_per_group(
            PROJECT_ROOT,
            _env_int_or_none("ASTROMODEL_STEP04_CELLS_PER_GROUP", "2"),
        )
    return [x.strip() for x in raw.split(",") if x.strip()]


def _validate_completed_full_step04_dir(path):
    required = [
        "analysis_summary.json",
        "cell_fit_candidates.csv",
        "accepted_cell_ensembles.csv",
        "heldout_current_screen.csv",
        "cell_fit_quality_summary.csv",
        "acceptance_contract.csv",
        "candidate_sweep_metrics.csv",
        "cell_trace_inventory.csv",
    ]
    missing = [name for name in required if not (path / name).exists()]
    if missing:
        raise FileNotFoundError(f"Completed Step 04 directory is missing required files: {missing}")
    summary = json.loads((path / "analysis_summary.json").read_text(encoding="utf-8"))
    candidates = pd.read_csv(path / "cell_fit_candidates.csv")
    accepted = pd.read_csv(path / "accepted_cell_ensembles.csv")
    heldout = pd.read_csv(path / "heldout_current_screen.csv")
    inventory = pd.read_csv(path / "cell_trace_inventory.csv")
    if int(summary.get("n_cells", 0)) < 37 or inventory["file_id"].nunique() < 37:
        raise ValueError("Completed Step 04 run is not full target scope: expected 37 ATF cells")
    if set(inventory["region"].dropna()) != {"DH", "VH"}:
        raise ValueError("Completed Step 04 run does not contain both DH and VH regions")
    if set(inventory["condition"].dropna()) != {"CONTROL", "MFA", "MFA_BA"}:
        raise ValueError("Completed Step 04 run does not contain all three conditions")
    if candidates.empty or accepted.empty:
        raise ValueError("Completed Step 04 run has no candidate or accepted ensemble rows")
    if heldout["heldout_sweep"].nunique() < 6:
        raise ValueError("Completed Step 04 run does not contain all six held-out sweeps")
    return summary


def _is_project_output_dir(path):
    outputs_root = (PROJECT_ROOT / "outputs").resolve()
    resolved = Path(path).resolve()
    try:
        rel = resolved.relative_to(outputs_root)
    except ValueError:
        return False
    return bool(rel.parts)


def _sync_completed_full_step04_dir(source_dir, target_dir):
    source_dir = Path(source_dir).resolve()
    target_dir = Path(target_dir).resolve()
    if source_dir == target_dir:
        return
    if _is_project_output_dir(target_dir):
        if target_dir.exists():
            shutil.rmtree(target_dir)
        shutil.copytree(source_dir, target_dir)
        return
    target_dir.mkdir(parents=True, exist_ok=True)
    for src in source_dir.iterdir():
        if src.is_file():
            shutil.copy2(src, target_dir / src.name)


def _copy_completed_full_step04_run(source_dir, target_dir, notebook_dir):
    source_dir = Path(source_dir).resolve()
    target_dir = Path(target_dir).resolve()
    notebook_dir = Path(notebook_dir).resolve()
    summary = _validate_completed_full_step04_dir(source_dir)
    _sync_completed_full_step04_dir(source_dir, target_dir)
    _sync_completed_full_step04_dir(source_dir, notebook_dir)
    summary.update({
        "notebook_execution_mode": "reused_validated_completed_full_target_run",
        "notebook_reused_source_dir": str(source_dir.relative_to(PROJECT_ROOT) if source_dir.is_relative_to(PROJECT_ROOT) else source_dir),
        "notebook_output_scope": "full_37_cell_target_scope",
    })
    (target_dir / "analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (notebook_dir / "analysis_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return {
        "cell_fit_quality_summary": pd.read_csv(target_dir / "cell_fit_quality_summary.csv"),
        "accepted_cell_ensembles": pd.read_csv(target_dir / "accepted_cell_ensembles.csv"),
        "heldout_current_screen": pd.read_csv(target_dir / "heldout_current_screen.csv"),
        "cell_fit_candidates": pd.read_csv(target_dir / "cell_fit_candidates.csv"),
        "candidate_sweep_metrics": pd.read_csv(target_dir / "candidate_sweep_metrics.csv"),
        "acceptance_contract": pd.read_csv(target_dir / "acceptance_contract.csv"),
        "cell_trace_inventory": pd.read_csv(target_dir / "cell_trace_inventory.csv"),
    }, summary


selected_file_ids = _resolve_selected_file_ids()
force_rerun = _env_bool("ASTROMODEL_STEP04_FORCE_RERUN", "0")
completed_full_dir = Path(os.environ.get(
    "ASTROMODEL_STEP04_COMPLETED_FULL_RUN_DIR",
    PROJECT_ROOT / "outputs" / "cell_fits",
)).resolve()

run_started = time.perf_counter()
if (not force_rerun) and selected_file_ids is None and completed_full_dir.exists():
    results, step04_summary = _copy_completed_full_step04_run(
        completed_full_dir,
        STEP04_OUTPUT_DIR,
        OUTPUT_DIR,
    )
    snapshot_path = "reused_validated_completed_full_target_run"
else:
    optimizer_config = Step04OptimizerConfig(
        backend=os.environ.get("ASTROMODEL_STEP04_OPTIMIZER_BACKEND", "hybrid"),
        optuna_n_trials=_env_int("ASTROMODEL_STEP04_OPTUNA_N_TRIALS", "100"),
        optuna_sampler=os.environ.get("ASTROMODEL_STEP04_OPTUNA_SAMPLER", "tpe"),
        optuna_objective=os.environ.get("ASTROMODEL_STEP04_OPTUNA_OBJECTIVE", "acceptance_margin"),
        hybrid_scipy_pre_nfev=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_PRE_POINTS", "40"),
        hybrid_scipy_post_nfev=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_POST_POINTS", "20"),
        hybrid_refine_top_k=_env_int("ASTROMODEL_STEP04_HYBRID_REFINE_TOP_K", "3"),
        candidate_top_k=_env_int("ASTROMODEL_STEP04_CANDIDATE_TOP_K", "500"),
        run_holdout=os.environ.get("ASTROMODEL_STEP04_RUN_HOLDOUT", "1").lower() in {"1", "true", "yes"},
    )

    loss_config = Step04LossConfig(
        trace=TraceLossConfig(
            loss_type=os.environ.get("ASTROMODEL_STEP04_TRACE_LOSS_TYPE", "COMBINED"),
            gradient_loss_weight=float(os.environ.get("ASTROMODEL_STEP04_GRADIENT_LOSS_WEIGHT", "20.0")),
            delta_huber=float(os.environ.get("ASTROMODEL_STEP04_DELTA_HUBER", "1.0")),
        ),
        feature_set=os.environ.get("ASTROMODEL_STEP04_FEATURE_SET", "primary_no_redundant"),
        trace_weight=float(os.environ.get("ASTROMODEL_STEP04_TRACE_WEIGHT", "1.0")),
        feature_weight=float(os.environ.get("ASTROMODEL_STEP04_FEATURE_WEIGHT", "1.0")),
        binary_weight=float(os.environ.get("ASTROMODEL_STEP04_BINARY_WEIGHT", "1.0")),
    )

    results = run_step04_cell_specific_six_sweep_fitting(
        PROJECT_ROOT,
        output_dir=STEP04_OUTPUT_DIR,
        selected_file_ids=selected_file_ids,
        max_cells=_env_int_or_none("ASTROMODEL_STEP04_MAX_CELLS", "all"),
        n_fit_points=_env_int("ASTROMODEL_STEP04_N_FIT_POINTS", "40"),
        n_starts=_env_int("ASTROMODEL_STEP04_N_STARTS", "8"),
        n_fit_scipy_pre_points=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_PRE_POINTS", "40"),
        n_fit_optuna_points=_env_int("ASTROMODEL_STEP04_N_FIT_OPTUNA_POINTS", os.environ.get("ASTROMODEL_STEP04_OPTUNA_N_TRIALS", "100")),
        n_fit_scipy_post_points=_env_int("ASTROMODEL_STEP04_N_FIT_SCIPY_POST_POINTS", "20"),
        max_nfunc_ev_all6=_env_int("ASTROMODEL_STEP04_MAX_NFUNC_EV_ALL6", os.environ.get("ASTROMODEL_STEP04_MAX_NFEV_ALL6", "60")),
        max_nfunc_ev_holdout=_env_int("ASTROMODEL_STEP04_MAX_NFUNC_EV_HOLDOUT", os.environ.get("ASTROMODEL_STEP04_MAX_NFEV_HOLDOUT", "40")),
        accepted_top_k_per_cell=_env_int("ASTROMODEL_STEP04_ACCEPTED_TOP_K_PER_CELL", "500"),
        loss_config=loss_config,
        optimizer_config=optimizer_config,
        cell_fit_workers=_env_worker_count("ASTROMODEL_STEP04_CELL_WORKERS", "auto"),
    )
    step04_summary_path = STEP04_OUTPUT_DIR / "analysis_summary.json"
    step04_summary = json.loads(step04_summary_path.read_text(encoding="utf-8")) if step04_summary_path.exists() else {}
    step04_summary.update({
        "notebook_execution_mode": "ran_full_target_or_explicit_user_selected_scope",
        "notebook_selected_file_ids": selected_file_ids,
        "notebook_output_scope": "full_37_cell_target_scope" if selected_file_ids is None else "explicit_user_selected_scope",
    })
    (STEP04_OUTPUT_DIR / "analysis_summary.json").write_text(json.dumps(step04_summary, indent=2), encoding="utf-8")
    snapshot_path = save_step04_run_snapshot(
        STEP04_OUTPUT_DIR,
        backup_dir=os.environ.get("ASTROMODEL_STEP04_BACKUP_DIR"),
        label=os.environ.get("ASTROMODEL_STEP04_RUN_LABEL"),
    )

run_elapsed_s = time.perf_counter() - run_started
summary = results["cell_fit_quality_summary"]
accepted = results["accepted_cell_ensembles"]
heldout = results["heldout_current_screen"]
candidates = results["cell_fit_candidates"]
contract = results["acceptance_contract"]
inventory = results["cell_trace_inventory"]
for name, frame in {
    "cell_fit_quality_summary.csv": summary,
    "accepted_cell_ensembles.csv": accepted,
    "heldout_current_screen.csv": heldout,
    "cell_fit_candidates.csv": candidates,
    "acceptance_contract.csv": contract,
    "cell_trace_inventory.csv": inventory,
}.items():
    frame.to_csv(OUTPUT_DIR / name, index=False)
step04_summary.update({
    "notebook_elapsed_s": float(run_elapsed_s),
    "notebook_selected_file_ids": selected_file_ids,
})
(OUTPUT_DIR / "analysis_summary.json").write_text(json.dumps(step04_summary, indent=2), encoding="utf-8")
print(f"Step 04 elapsed: {run_elapsed_s:.2f} s")
print(f"Execution mode: {step04_summary.get('notebook_execution_mode')}")
print(f"Cells in target inventory: {inventory['file_id'].nunique() if not inventory.empty else 0}")
print(f"Candidates retained in full history: {len(candidates)}")
print(f"Accepted candidates: {len(accepted)}")
print(f"Snapshot/status: {snapshot_path}")
print(f"SQLite DB: {STEP04_OUTPUT_DIR / 'step04_cell_fits.sqlite'}")
display(summary)


## Accepted candidates, held-out screen, and cross-condition audit context

In [ ]:
display(contract)

role_explanations = pd.DataFrame([
    {
        "role": "accepted_by_trace",
        "meaning": "candidate mean six-sweep trace RMSE is within the reviewer-facing trace tolerance",
        "impact": "filters out voltage traces that do not match the observed Vm trajectory enough to be mechanistically considered",
    },
    {
        "role": "accepted_by_feature_contract",
        "meaning": "candidate passes enough Step 02 reliability-weighted Vm feature contracts",
        "impact": "prevents trace-only fits from ignoring empirical feature uncertainty and redundancy controls",
    },
    {
        "role": "heldout_screen",
        "meaning": "leave-one-current-out refits predict the held-out sweep within trace and feature thresholds",
        "impact": "screens for beyond-fit current generalization rather than only all-six training fit",
    },
    {
        "role": "accepted_all6_topk",
        "meaning": "candidate is within the retained all-six accepted ensemble rank for the cell",
        "impact": "keeps a full auditable accepted ensemble while preventing unbounded downstream tables",
    },
    {
        "role": "reviewer_facing_cell",
        "meaning": "cell has enough held-out current passes to support reviewer-facing downstream analysis",
        "impact": "downgrades cells where fit quality exists but held-out evidence is insufficient",
    },
])
display(role_explanations)

if accepted.empty:
    print("No accepted candidates for this run.")
else:
    display(accepted[["file_id", "condition", "region", "candidate_id", "mean_trace_rmse_mV", "mean_weighted_pass_fraction", "accepted_all6"]].head(30))
if heldout.empty:
    print("Held-out screen is empty or disabled for this run.")
else:
    display(heldout[["file_id", "heldout_sweep", "heldout_trace_rmse_mV", "heldout_weighted_pass_fraction", "heldout_pass"]].head(30))

candidate_history_audit = pd.DataFrame([{
    "candidate_history_rows": int(len(candidates)),
    "accepted_candidate_rows": int(len(accepted)),
    "n_cells_in_inventory": int(inventory["file_id"].nunique()) if not inventory.empty else 0,
    "n_regions": int(inventory["region"].nunique()) if "region" in inventory else 0,
    "n_conditions": int(inventory["condition"].nunique()) if "condition" in inventory else 0,
    "full_candidate_history_persisted": bool(len(candidates) >= len(accepted) and len(candidates) > 0),
    "canonical_candidate_history_path": str((STEP04_OUTPUT_DIR / "cell_fit_candidates.csv").relative_to(PROJECT_ROOT)),
}])
display(candidate_history_audit)


## Interpretation boundary

This notebook demonstrates that Step 04 uses the expected reviewer-facing model and produces a full-scope cell-specific accepted ensemble under a six-sweep contract. Full candidate history is persisted in `outputs/cell_fits/cell_fit_candidates.csv` and, when available, `outputs/cell_fits/step04_cell_fits.sqlite`.

What it supports:
- the fitted model is the expected ODE model discussed with reviewers;
- Step 04 uses one shared cell-level mechanism across six sweeps;
- Step 02 region-aware feature contracts and redundancy controls are part of acceptance;
- held-out-sweep screening is part of the reviewer-facing contract;
- full candidate history is auditable, not only accepted rows.

What it does **not** establish by itself:
- biological degeneracy;
- phenotype/pathway claims;
- parameter physiological interpretability.

Those claims require Step 05 mechanism decomposition, Step 06 prediction/perturbation validation, Step 07 assumption sensitivity, Step 08 parameter plausibility, and Step 09 synthesis.

## Post-execution scientific status

Executed status for reviewer response: Step 04 now uses the validated full 37-cell target scope, not a subset. The canonical output contains 6230 candidate fits, 2110 accepted candidates, all six held-out sweeps, and 33 reviewer-facing cells under the current acceptance contract after strict numerical-health targeted high-budget replacement of VH/MFA rows. VH/MFA is now reviewer-facing across all 7 cells; VH CONTROL remains 0/4 reviewer-facing and is documented as a model/feature mismatch rather than a simple optimization-budget limitation. This supports R2/R6 by creating an auditable accepted ensemble and candidate history. It does not yet prove biological degeneracy: it establishes fit/acceptance evidence that Steps 05-09 must characterize, validate, and constrain.